In [ ]:
# Put import statements here
import os
# hide tensorflow info/warning logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["KERAS_BACKEND"] = "torch"
import sys
import subprocess
from pathlib import Path
import json
import pandas as pd


# Local files/code

import src.data_preprocessing.text_preprocessing as text_pre
from src.agents.Visual_Agent import VisualModel
from src.util.logger import Logger
import src.config as config
import src.util.general as general_util
from src.data_preprocessing.TextTokenizer import TextTokenizer
from src.data_preprocessing.TextEmbedder import TextEmbedder
from src.data_preprocessing.USDA_processing import embed_USDA_data
import src.evaluation.visual_evaluator as visual_evaluator
import src.data_preprocessing.image_preprocessing as img_pre
#from src.agents.NLP_Agent import AnswerRanker, build_answer_index, add_usda_records
#import src.evaluation.text_evaluator as text_eval
from src.data_preprocessing.build_training_sets import load_vqa_dataset

In [ ]:
#load in the GBIF data
#GET THE PATH TO ROOT OF PROJECT
ROOT_PATH = Path.cwd()
print(ROOT_PATH)
TRAIN_DATA = pd.read_csv(config.Data.GBIF.TRAIN_FILE)
TEST_DATA = pd.read_csv(config.Data.GBIF.TEST_FILE)
VAL_DATA = pd.read_csv(config.Data.GBIF.VALIDATION_FILE)

In [ ]:
updated = pd.read_csv("GBIF_UPDATED_SEVERITY.csv").set_index("img_id")
for frame in (TRAIN_DATA, VAL_DATA, TEST_DATA):
    frame["disease"] = frame["image_id"].map(updated["disease"]).fillna(frame["disease"])
    frame["severity"] = frame["image_id"].map(updated["severity"]).fillna(frame["severity"])

In [ ]:
VQA_TRAIN = pd.read_csv(config.Data.PlantExpertVQA.TRAIN_FILE)
VQA_VAL = pd.read_csv(config.Data.PlantExpertVQA.VALIDATION_FILE)
VQA_TEST = pd.read_csv(config.Data.PlantExpertVQA.TEST_FILE)
VQA_ROOT_PATH = config.Data.PlantExpertVQA.ROOT.relative_to(config.General.PROJECT_ROOT)
for frame in (VQA_TRAIN, VQA_VAL, VQA_TEST):
    frame["image_path"] = frame["image_path"].apply(lambda p: str(VQA_ROOT_PATH / p))
    frame["genus"] = "unknown"
    frame["family"] = "unknown"
    frame["symbol"] = "unknown"

In [ ]:
##Iterate Through Data sets and fix pathing issue

TRAIN_DATA["image_path"] = TRAIN_DATA["image_path"].apply(lambda x: x.lstrip('//'))
#TRAIN_DATA["image_path"] = TRAIN_DATA["image_path"].apply(lambda p: str(ROOT_PATH / p))


TEST_DATA["image_path"] = TEST_DATA["image_path"].apply(lambda x: x.lstrip('//'))
#TEST_DATA["image_path"] = TEST_DATA["image_path"].apply(lambda p: str(ROOT_PATH / p))

VAL_DATA["image_path"] = VAL_DATA["image_path"].apply(lambda x: x.lstrip(r'/'))
#VAL_DATA["image_path"] = VAL_DATA["image_path"].apply(lambda p: str(ROOT_PATH / p))

print(VAL_DATA.columns)


In [ ]:
for i, r in TRAIN_DATA.iterrows():
    print(r["image_path"])

In [ ]:
traits = ["crop", "disease", "severity", "genus", "family", "symbol"]


train_img = pd.concat([
    VisualModel.one_row_per_image(TRAIN_DATA, traits),
    VisualModel.one_row_per_image(VQA_TRAIN, traits)
], ignore_index=True)
val_img = pd.concat([
    VisualModel.one_row_per_image(VAL_DATA, traits),
    VisualModel.one_row_per_image(VQA_VAL, traits)
], ignore_index=True)
test_img = pd.concat([
    VisualModel.one_row_per_image(TEST_DATA, traits),
    VisualModel.one_row_per_image(VQA_TEST, traits)
])
classes = VisualModel.build_classes(train_img, traits)

IMG_SIZE = (224, 224)
train_ds = VisualModel.make_dataset(train_img, classes, config.General.PROJECT_ROOT, training=True, use_mask=True, size = IMG_SIZE)
val_ds  = VisualModel.make_dataset(val_img, classes, config.General.PROJECT_ROOT, use_mask=True, size = IMG_SIZE)



In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch
print(keras.backend.backend())
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
keras.mixed_precision.set_global_policy("mixed_bfloat16")

In [ ]:

visual_model = VisualModel(classes=classes, img_size=IMG_SIZE, use_mask=True)
visual_model.build()
visual_model.compile()

history_frozen = visual_model.fit(train_ds, val_ds, epochs=30)
visual_model.unfreeze_layers(n_layers=100)
history_tuned = visual_model.fit(train_ds, val_ds, epochs=5)

visual_evaluator.plot_loss(history_frozen, history_tuned)
visual_evaluator.plot_accuracy(history_frozen, history_tuned, heads=["crop", "disease", "severity", "genus", "family", "symbol"])

import json
Path("./models/visual_models/10_11_classes.json").write_text(json.dumps(classes))
visual_model.save(path="./models/visual_models/10_11.keras")
 

In [ ]:
# GRID SWEEP ON HYPERPARAMETERS
dropout_options = [.3, .5]
lr_options = [1e-3, 1e-4]
batch_size_options = [32]
size_options = [112, 224]
mask_options = [True, False]

FROZEN_EPOCHS = 30
UNFROZEN_EPOCHS = 5
NUM_LAYERS = 100
TUNE_LR = 1e-5

training_loss_weights = {class_head : 1.0 for class_head in classes}
training_loss_weights["disease"] = 3.0
training_loss_weights["severity"] = 3.0

Path("./models/visual_models/sweep").mkdir(parents=True, exist_ok=True)
Path("./models/visual_models/sweep/classes.json").write_text(json.dumps(classes))

results = []
num_trials = 1
sweep_train = train_img.sample(min(40000, len(train_img)), random_state=0)
sweep_val = val_img.sample(min(8000, len(val_img)), random_state=0)

for batch_size in batch_size_options:
    for lr_option in lr_options:
        for dropout_option in dropout_options:
            for size_option in size_options:
                for mask_option in mask_options:
                    image_size = (size_option, size_option)
                    print(f"\n----------\nTrial {num_trials}\nconfiguration:\nbatch size: {batch_size}\nlearning rate\n{lr_option}\ndropout{dropout_option}\nimage size: {image_size}\nUsing mask: {mask_option}\n----------")

                    train_ds = VisualModel.make_dataset(
                        sweep_train, classes, config.General.PROJECT_ROOT, size=image_size,
                        batch_size=batch_size, training=True, use_mask=mask_option
                    )

                    val_ds = VisualModel.make_dataset(
                        sweep_val, classes, config.General.PROJECT_ROOT, size=image_size,
                        batch_size=batch_size, use_mask=mask_option
                    )

                    sweep_visual_model = VisualModel(
                        classes=classes,
                        img_size=image_size,
                        dropout=dropout_option,
                        loss_weights=training_loss_weights,
                        use_mask=mask_option)


                    sweep_visual_model.build()
                    sweep_visual_model.compile(lr=lr_option)

                    checkpoint = f"./models/visual_models/sweep/trial_{num_trials}.keras"

                    history_frozen = sweep_visual_model.fit(
                        train_ds, val_ds, epochs=FROZEN_EPOCHS,
                        checkpoint_model=checkpoint
                    )

                    sweep_visual_model.unfreeze_layers(n_layers=NUM_LAYERS, lr=TUNE_LR)

                    history_unfrozen = sweep_visual_model.fit(
                        train_ds,
                        val_ds,
                        epochs=UNFROZEN_EPOCHS,
                        checkpoint_model=checkpoint
                    )

                    frozen_best = max(history_frozen.history["val_crop_accuracy"])
                    unfrozen_best = max(history_unfrozen.history["val_crop_accuracy"])

                    results.append({
                        "trial": num_trials,
                        "batch_size": batch_size,
                        "lr": lr_option,
                        "dropout": dropout_option,
                        "image_size": size_option,
                        "use_mask": mask_option,
                        "frozen_best": frozen_best,
                        "unfrozen_best": unfrozen_best,
                        "epochs_frozen_ran": len(history_frozen.history["val_crop_accuracy"]),
                        "checkpoint": checkpoint,
                    })
                    print(
                        f"\ntrial {num_trials}:\n"
                        f"(frozen best: {frozen_best:.4f}, unfrozen best: {unfrozen_best:.4f})"
                    )

                    pd.DataFrame(results).to_csv(
                        "./models/visual_models/sweep/results.csv", index=False)

                    del sweep_visual_model, train_ds, val_ds
                    keras.backend.clear_session()
                    torch.cuda.empty_cache()
                    num_trials += 1

sweep_results = pd.DataFrame(results).sort_values("unfrozen_best", ascending=False)
print("\n" + sweep_results.to_string(index=False))

In [ ]:
#EVALUATION OF BEST MODEL
model_path = "./models/visual_models/10_11.keras" #best_model["checkpoint"]
classes_path = "./models/visual_models/10_11_classes.json"
#best_model = sweep_results.iloc[0]
visual_model_best = VisualModel.load(model_path= model_path, classes_path=classes_path) #img_size=(int(best_model["image_size"]),)*2)
#Path("./models/visual_models/visual_model_best_classes.json").write_text(json.dumps(classes))
#visual_model_best.save(path="./models/visual_models/visual_model_best.keras")
visual_model_best.model.summary()
test_predictions = visual_evaluator.predict(visual_model_best, test_img, config.General.PROJECT_ROOT)
print(visual_evaluator.metrics_table(test_predictions, num_result=3).to_string())
visual_evaluator.print_report(test_predictions, visual_model_best.classes, "severity")
visual_evaluator.plot_confusion_matrix(test_predictions, visual_model_best.classes, "severity")

In [ ]:

FROZEN_EPOCHS = 30
UNFROZEN_EPOCHS = 5
NUM_LAYERS = 100
TUNE_LR = 1e-5

best = (pd.read_csv("./models/visual_models/sweep/results.csv").sort_values("unfrozen_best", ascending=False).iloc[0])
print(f"best hyperparameters: {best}")

BEST_SIZE = (int(best["image_size"]),) * 2
BEST_BATCH = int(best["batch_size"])
BEST_MASK = str(best["use_mask"]).strip().lower() == "true"

train_ds_full = VisualModel.make_dataset(
    train_img,
    classes,
    config.General.PROJECT_ROOT,
    size=BEST_SIZE,
    batch_size=BEST_BATCH,
    training=True,
    use_mask=BEST_MASK
)

val_ds_full = VisualModel.make_dataset(
    val_img,
    classes,
    config.General.PROJECT_ROOT,
    size=BEST_SIZE,
    batch_size=BEST_BATCH,
    use_mask=BEST_MASK
)

final_model = VisualModel(
    classes=classes,
    img_size=BEST_SIZE,
    dropout=float(best["dropout"]),
    use_mask=BEST_MASK
)

final_model.build()
final_model.compile(lr=float(best["lr"]))

best_checkpoint = "./models/visual_models/final_best.keras"
history_frozen = final_model.fit(
    train_ds_full, val_ds_full, epochs=FROZEN_EPOCHS,
    checkpoint_model=best_checkpoint
)
final_model.unfreeze_layers(n_layers=NUM_LAYERS, lr=TUNE_LR)


In [ ]:
tuned_checkpoint = "./models/visual_models/final_best_tuned.keras"
history_tuned = final_model.fit(
    train_ds_full, val_ds_full, epochs=UNFROZEN_EPOCHS,
    checkpoint_model=tuned_checkpoint
)

final_model.save("./models/visual_models/final.keras")
Path("./models/visual_models/final_classes.json").write_text(json.dumps(classes))


In [ ]:
final_model.model.summary()
test_predictions = visual_evaluator.predict(final_model, test_img, config.General.PROJECT_ROOT)
print(visual_evaluator.metrics_table(test_predictions, num_result=3).to_string())
visual_evaluator.plot_loss(history_frozen, history_tuned)
visual_evaluator.plot_accuracy(history_frozen, history_tuned, heads=["crop", "disease", "severity", "genus", "family", "symbol"])
visual_evaluator.print_report(test_predictions, final_model.classes, "severity")
visual_evaluator.plot_confusion_matrix(test_predictions, final_model.classes, "severity")